# Pipeline example with simple MLP layers

## What you need to know for this tutorial
1. A basic knowledge of Jax and numpy
2. Familiarity with the concepts of tensor sharding and parallelism
3. Familiarity with basic neural network concepts like dense layers, activation functions, and backprop

## Optional, but helpful, to know
1. Flax package for building composable, trainable models in Jax
2. Optax package with different optimizers

## Initialize Legate-Jax
The first step in a Legate-Jax program is to initialize the environment. Before any other imports, `legate.jax.init` should be called with the appropriate arguments for configuring Legate. This does violate the common Python style rule of top-level imports coming before all code. It is possible to configure Legate ahead-of-time with environment variables, but the programmatic method here is generally cleaner and easier.  The parameters for Legate must be configured before importing any Jax functions.

In [2]:
from jax_plugins.legate import init

init(
  cpus=8,
)

## Import Jax packages

Once Legate-Jax has been initialized, the standard set of Jax imports can be done. In this case, we are also importing:
1. Flax, which is used to define modules with parameters
1. Optax, which provides optimizers for updating parameters
1. Jax sharding APIs for defining parallel computations

For more details on [Flax](https://github.com/google/flax) and [Optax](https://github.com/google-deepmind/optax), the user should consult the docs for these packages. For this tutorial, the usage of these packages should be relatively clear.

In [3]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
from jax.sharding import Mesh, PartitionSpec as P
from legate.jax import shard_axes, with_sharding_constraint
from flax import linen as nn
import legate.jax

## Create single layer with sharding annotations

We can now proceed to define a model called `Pipeline`. The building block for this model will be a basic MLP layer which is a `nn.Dense` followed by a `nn.relu`. The dense layer implicitly defines a parameter tensor. Two key observations:

1. A sharding constraint is defined for the input activations `x` that names the tensor axes. The input activations are 1D with the axis named `batch`.
2. Sharding is also defined for the 2D tensor in the dense layer. We give these axes the names `data` and `model`.

In [4]:
class MLPBlock(nn.Module):
    features: int = 4
    @nn.compact
    def __call__(self, x):
        x = with_sharding_constraint(x, P("batch"))
        x = nn.Dense(features=self.features,kernel_init=shard_axes("data","model"))(x)
        x = nn.relu(x)
        return x

## Create full model with pipelined layers

Using the MLP building blocks, we can build up a pipeline parallel model. Each layer of the pipeline will be assigned to a different submesh, which is computed in the `submesh` method.
For each new layer, the device grid is incremented. An important abstraction here is the `axis_map` used in the `Pipeline`. 
Legate-Jax encourages (but does not require) tensor axes to be given logical, descriptive names rather than positional or hardware-specific names. This follows the philosophy used in [Levanter](https://crfm.stanford.edu/2023/06/16/levanter-1_0-release.html) outlined in this [article](https://nlp.seas.harvard.edu/NamedTensor) - and also followed by [maxtext](https://github.com/google/maxtext) and [T5x](https://github.com/google-research/t5x) and their use of [logical sharding rules](https://flax.readthedocs.io/en/v0.8.2/api_reference/flax.linen/_autosummary/flax.linen.logical_to_mesh_axes.html).  Flax code is written with logical names (`batch`, `data`, `model`), which then must be bound to physical device axes (`x` and `y`) to fully define the parallelism. This creates a separation between model specification (logical code written with named tensor axes) and model mapping (physical sharding of axes across a device mesh).

In this particular case, we map the `batch` and `data` logical dimensions to the physical `x` axis and give each submesh a `4x1` shape. Legate-Jax therefore provides the following capabilities:
1. The ability to assign computations to a submesh within a larger computation. Jax enforces an SPMD requirement where tensors must be sharded across all devices and every device must participate.
2. Logical sharding rules within a submesh. This provides similar functionality to T5x and Levanter, but allows arbitrary resharding *within* a submesh.

In [5]:
class Pipeline(nn.Module):
    n_layers: int = 2
    shape: tuple[int] = (4,1)
    axes = ("x", "y")
    axis_map = (
      ("batch", "x"),
      ("data", "x"),
      ("model", "y"),
    )

    def submesh(self, layer: int) -> Mesh:
        devices = np.array(jax.devices())
        submesh_size = np.prod(self.shape)
        off = layer * submesh_size
        device_grid = devices[off:off+submesh_size].reshape(*self.shape)
        return Mesh(device_grid, self.axes)

    @nn.compact
    def __call__(self, x):
        for i in range(self.n_layers):
            layer = legate.jax.task(MLPBlock, mesh=self.submesh(i), name=f"layer_{i}", logical_axes=self.axis_map)()
            x = layer(x)

        return (x*x).sum()

## Initialize parameters and create parallel training function

So far the code looks almost the same as standard Flax code with a few extra annotations. The full power of Legate-Jax comes from its auto-parallelizing compiler that reads all the submesh annotations and transforms the global computation into a series of submesh tasks. Managing Jax training state and shardings can be difficult when directly calling `jax.jit`. Legate-Jax provides a `parallelize_step` for training that allows users to write parallelism-agnostic code for a given input model. The user simply passes in the `model`, a standard `optax` optimizer, and an example input batch. Legate-Jax does all the work to derive all the shardings, compile the model, and initialize all the sharded training state. `parallelize_step` returns back the four objects necessary to run a training loop:

1. A `step_fn` for computing gradients and updating parameters
2. The randomly initialized sharded parameters
3. A function for preprocessing input batches located in host memory into device arrays
4. The mesh context to use for traing steps

In [6]:
from legate.jax import parallelize_step
x = jnp.arange(16)
opt = optax.sgd(learning_rate=0.02)
step_fn, sharded_train_state, prepare_batch, mesh = parallelize_step(model=Pipeline(), optimizer=opt, batch=x)

Platform 'legate' is experimental and not all JAX functionality may be correctly supported!
I0000 00:00:1724542096.397864  772186 cpu_client.cc:466] TfrtCpuClient created.


FLAX 1
WTAF:  DynamicJaxprTrace(level=1/1)


AttributeError: DynamicJaxprTracer has no attribute _state

## Run training steps

Now that we have our initial training state (parameters + optimizer state), we can run training steps to compute the loss and increment parameters.

In [ ]:
with mesh:
    batch = prepare_batch(x)
    loss, sharded_train_state = step_fn(sharded_train_state, batch)
    print("Loss = ", loss)

## Deep dive: MPMD sharding specs

To get a glimpse of the underlying details, we can inspect the sharding specs of the training state. There are 5 tensors: 
1. A single scalar parameter that is not partitioned (empty `PartitionSpec()`). This is the learning rate (optimizer state for SGD).
2. A (4,) tensor replicated across devices 0-3 corresponding to the first layer bias
3. A (16,4) tensor fully sharded across devices 0-3 corresponding to the fully connected weights in the first layer
4. A (4,) tensor replicated across devices 4-7 corresponding to the second layer bias
5. A (16,4) tensor fully sharded across devices 4-7 corresponding to the fully connected weights in the second layer

In [ ]:
jax.tree_map(lambda x: print(x.shape, x.sharding), sharded_train_state)